# 01 — Exploración de datos

Análisis exploratorio del dataset de cabecera del gasoducto: un mes de operación normal con muestreo cada 1 minuto, presión (P) y caudal (Q).

**Objetivo:** caracterizar la calidad y estructura de los datos, y derivar empíricamente las propiedades que justifican el diseño del detector en capas (sanidad, suavizado, divergencia P-Q, CUSUM).

**Dataset:** `data/PLOT_TS_P_Q.csv` (no versionado en el repo).

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Configuración de matplotlib
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

# Ruta al CSV (relativa a la ubicación del notebook)
DATA_PATH = Path("..") / "data" / "PLOT_TS_P_Q.csv"
assert DATA_PATH.exists(), f"No se encuentra el CSV en {DATA_PATH.resolve()}"

print(f"CSV encontrado en: {DATA_PATH.resolve()}")
print(f"Tamaño: {DATA_PATH.stat().st_size / 1024:.1f} KB")

CSV encontrado en: C:\Users\agust\Portfolio\monitor-gasoducto\data\PLOT_TS_P_Q.csv
Tamaño: 1634.1 KB


## 1. Carga y primer vistazo

Lectura del CSV con separador `;` y separador decimal `.`. Parseo de la columna `TS` con formato `dd/mm/yyyy HH:MM:SS`. Inspección inicial de estructura, tipos y rango.

In [ ]:
# Carga del CSV
df = pd.read_csv(
    DATA_PATH,
    sep=";",
    parse_dates=["TS"],
    date_format="%d/%m/%Y %H:%M:%S",
)

# Primer vistazo
print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nRango temporal: {df['TS'].min()}  →  {df['TS'].max()}")
print(f"\nPrimeras 5 filas:")
df.head()

## 2. Sanidad temporal

Verificación de propiedades estructurales de la serie temporal: orden, duplicados, gaps en el muestreo y valores faltantes en P/Q. Estos chequeos son precondición de todo el pipeline del detector (suavizado, derivadas, CUSUM asumen secuencia ordenada y regular).

In [ ]:
# 2.1 — ¿Está ordenado cronológicamente?
is_sorted = df["TS"].is_monotonic_increasing
print(f"TS ordenado cronológicamente: {is_sorted}")

# 2.2 — ¿Hay duplicados de TS?
n_duplicados = df["TS"].duplicated().sum()
print(f"Timestamps duplicados: {n_duplicados}")

# 2.3 — ¿NaN en P o Q?
print(f"\nNaN por columna:")
print(df[["P", "Q"]].isna().sum())

# 2.4 — Análisis de la cadencia (intervalos entre muestras consecutivas)
deltas = df["TS"].diff().dropna()
print(f"\nEstadísticas de los intervalos entre muestras:")
print(f"  Mínimo:  {deltas.min()}")
print(f"  Máximo:  {deltas.max()}")
print(f"  Mediana: {deltas.median()}")
print(f"  Moda:    {deltas.mode().iloc[0]}")

# 2.5 — Distribución de los intervalos (cuántos son "anómalos")
print(f"\nIntervalos únicos y sus frecuencias (top 10):")
print(deltas.value_counts().head(10))

# 2.6 — ¿Cuántas muestras faltantes implican los gaps?
intervalo_esperado = pd.Timedelta(minutes=1)
muestras_faltantes = ((deltas - intervalo_esperado) / intervalo_esperado).clip(lower=0).sum()
print(f"\nMuestras faltantes (estimadas a partir de gaps): {int(muestras_faltantes)}")

In [ ]:
# Localización del gap único en el tiempo
intervalo_esperado = pd.Timedelta(minutes=1)
deltas = df["TS"].diff()

# Filas posteriores a un gap (delta > 1 min)
idx_post_gap = df.index[deltas > intervalo_esperado]

print(f"Cantidad de gaps detectados: {len(idx_post_gap)}")

for idx in idx_post_gap:
    ts_pre = df.loc[idx - 1, "TS"]
    ts_post = df.loc[idx, "TS"]
    duracion = ts_post - ts_pre

    # Posición relativa dentro del rango total
    ts_inicio = df["TS"].iloc[0]
    ts_fin = df["TS"].iloc[-1]
    pos_relativa = (ts_pre - ts_inicio) / (ts_fin - ts_inicio) * 100

    print(f"\nGap detectado:")
    print(f"  Último dato pre-gap : {ts_pre}")
    print(f"  Primer dato post-gap: {ts_post}")
    print(f"  Duración del gap    : {duracion}")
    print(f"  Posición relativa   : {pos_relativa:.1f}% del mes")

## 3. Distribuciones marginales

Inspección por separado de P y Q: estadísticas descriptivas (rango operativo, dispersión, percentiles) e histogramas (forma, modalidad, colas, valores anómalos). Estas propiedades determinan si la regresión Q=f(P) de la capa 2 puede modelarse como una sola relación lineal o si hay regímenes operativos distintos.

In [ ]:
# Estadísticas descriptivas
print("Estadísticas descriptivas de P y Q:")
print(df[["P", "Q"]].describe())

In [ ]:
# Histogramas de P y Q
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["P"], bins=80, edgecolor="black", linewidth=0.3)
axes[0].set_title("Distribución de P (presión)")
axes[0].set_xlabel("P")
axes[0].set_ylabel("Frecuencia")
axes[0].grid(alpha=0.3)

axes[1].hist(df["Q"], bins=80, edgecolor="black", linewidth=0.3)
axes[1].set_title("Distribución de Q (caudal)")
axes[1].set_xlabel("Q")
axes[1].set_ylabel("Frecuencia")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Series temporales

Visualización de P y Q a lo largo del tiempo a tres escalas: mes completo (tendencias y regímenes), una semana (patrones diarios) y un día (cuantización del SCADA y variabilidad fina).

In [ ]:
# 4.1 — Vista global: el mes completo
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

axes[0].plot(df["TS"], df["P"], linewidth=0.5)
axes[0].set_ylabel("P (presión)")
axes[0].set_title("Mes completo — Presión")
axes[0].grid(alpha=0.3)

axes[1].plot(df["TS"], df["Q"], linewidth=0.5, color="tab:orange")
axes[1].set_ylabel("Q (caudal)")
axes[1].set_xlabel("Tiempo")
axes[1].set_title("Mes completo — Caudal")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 4.2 — Vista intermedia: una semana en el medio del mes
ts_inicio = df["TS"].iloc[0]
ventana_inicio = ts_inicio + pd.Timedelta(days=7)
ventana_fin = ventana_inicio + pd.Timedelta(days=7)

mascara_semana = (df["TS"] >= ventana_inicio) & (df["TS"] < ventana_fin)
df_semana = df[mascara_semana]

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

axes[0].plot(df_semana["TS"], df_semana["P"], linewidth=0.7)
axes[0].set_ylabel("P (presión)")
axes[0].set_title(f"Semana {ventana_inicio.date()} → {ventana_fin.date()} — Presión")
axes[0].grid(alpha=0.3)

axes[1].plot(df_semana["TS"], df_semana["Q"], linewidth=0.7, color="tab:orange")
axes[1].set_ylabel("Q (caudal)")
axes[1].set_xlabel("Tiempo")
axes[1].set_title(f"Semana {ventana_inicio.date()} → {ventana_fin.date()} — Caudal")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 4.3 — Vista fina: un día en el medio de la semana anterior
dia_inicio = ventana_inicio + pd.Timedelta(days=3)
dia_fin = dia_inicio + pd.Timedelta(days=1)

mascara_dia = (df["TS"] >= dia_inicio) & (df["TS"] < dia_fin)
df_dia = df[mascara_dia]

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

axes[0].plot(df_dia["TS"], df_dia["P"], linewidth=1.0, marker=".", markersize=2)
axes[0].set_ylabel("P (presión)")
axes[0].set_title(f"Día {dia_inicio.date()} — Presión (con marcadores por muestra)")
axes[0].grid(alpha=0.3)

axes[1].plot(df_dia["TS"], df_dia["Q"], linewidth=1.0, color="tab:orange", marker=".", markersize=2)
axes[1].set_ylabel("Q (caudal)")
axes[1].set_xlabel("Tiempo")
axes[1].set_title(f"Día {dia_inicio.date()} — Caudal (con marcadores por muestra)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Relación P-Q

Análisis de la dependencia entre presión y caudal: scatter plot para inspeccionar la forma de la nube (lineal, curva, multi-modal), correlación de Pearson (relación lineal) y de Spearman (relación monótona). Esta sección determina si la regresión Q=f(P) de la capa 2 puede modelarse como una sola recta o si la estructura es más compleja.

In [ ]:
# Scatter plot P vs Q
fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(df["P"], df["Q"], alpha=0.1, s=4)
ax.set_xlabel("P (presión)")
ax.set_ylabel("Q (caudal)")
ax.set_title("Relación P–Q (todo el mes)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlaciones
corr_pearson = df[["P", "Q"]].corr(method="pearson").iloc[0, 1]
corr_spearman = df[["P", "Q"]].corr(method="spearman").iloc[0, 1]

print(f"Correlación de Pearson  (lineal):   {corr_pearson:.4f}")
print(f"Correlación de Spearman (monótona): {corr_spearman:.4f}")
print(f"\nDiferencia (Spearman - Pearson):    {corr_spearman - corr_pearson:+.4f}")

## 6. Deadband del SCADA

Caracterización de la cuantización del SCADA: el sensor solo reporta cambios cuando superan un umbral mínimo (deadband), por lo que valores cuasi-constantes son la regla y los saltos discretos son la excepción. Esto invalida el uso de derivadas crudas (la mayoría serán cero) y justifica la capa 1 (suavizado) del detector.

**Dato operativo conocido:** el deadband nominal de P es 0.2.
El de Q se estima empíricamente desde los datos.

In [ ]:
# Saltos minuto a minuto en P y Q (valor absoluto)
saltos_P = df["P"].diff().abs()
saltos_Q = df["Q"].diff().abs()

# Filtramos los ceros (filas sin cambio)
saltos_P_no_cero = saltos_P[saltos_P > 0]
saltos_Q_no_cero = saltos_Q[saltos_Q > 0]

total = len(df) - 1  # diff() pierde una fila

print("Frecuencia de cambio (cuantización):")
print(f"  P cambia en {len(saltos_P_no_cero):>6} de {total} filas ({100*len(saltos_P_no_cero)/total:.2f}%)")
print(f"  Q cambia en {len(saltos_Q_no_cero):>6} de {total} filas ({100*len(saltos_Q_no_cero)/total:.2f}%)")

In [ ]:
# Estadísticas de los saltos no-cero
print("Saltos no-cero — estadísticas:")
print()
print("P:")
print(saltos_P_no_cero.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print(f"  Moda: {saltos_P_no_cero.mode().iloc[0]}")
print()
print("Q:")
print(saltos_Q_no_cero.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print(f"  Moda: {saltos_Q_no_cero.mode().iloc[0]}")

In [ ]:
# Histogramas de los saltos (zoom en valores chicos donde está la cuantización)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# P: zoom hasta el percentil 95 para ver la estructura de la cuantización
p_max_plot = saltos_P_no_cero.quantile(0.95)
axes[0].hist(saltos_P_no_cero[saltos_P_no_cero <= p_max_plot], bins=200, edgecolor="black", linewidth=0.2)
axes[0].axvline(0.2, color="red", linestyle="--", linewidth=1, label="Deadband nominal (0.2)")
axes[0].set_title("Saltos no-cero de P (zoom hasta percentil 95)")
axes[0].set_xlabel("|ΔP|")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Q: zoom hasta el percentil 95
q_max_plot = saltos_Q_no_cero.quantile(0.95)
axes[1].hist(saltos_Q_no_cero[saltos_Q_no_cero <= q_max_plot], bins=200, edgecolor="black", linewidth=0.2, color="tab:orange")
axes[1].set_title("Saltos no-cero de Q (zoom hasta percentil 95)")
axes[1].set_xlabel("|ΔQ|")
axes[1].set_ylabel("Frecuencia")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Reporte comparativo: deadband estimado vs rango operativo
deadband_P_estimado = saltos_P_no_cero.min()
deadband_Q_estimado = saltos_Q_no_cero.min()

rango_P = df["P"].max() - df["P"].min()
rango_Q = df["Q"].max() - df["Q"].min()

print("Deadband estimado vs rango operativo:")
print(f"  P: deadband = {deadband_P_estimado:.4f} | rango = {rango_P:.2f} | relativo = {100*deadband_P_estimado/rango_P:.4f}%")
print(f"  Q: deadband = {deadband_Q_estimado:.4f} | rango = {rango_Q:.2f} | relativo = {100*deadband_Q_estimado/rango_Q:.4f}%")

print(f"\nVerificación contra dato operativo:")
print(f"  Deadband nominal de P: 0.2")
print(f"  Deadband empírico de P: {deadband_P_estimado:.4f}")
print(f"  Diferencia: {abs(deadband_P_estimado - 0.2):.4f}")

### 6.1 — Verificación del heartbeat horario (P)

Hipótesis (confirmada operativamente por el dato del SCADA): el sensor de P aplica deadband de 0.2 pero **fuerza una actualización cada hora completa** (heartbeat al cumplirse `hh:00:00`). Esto explica los saltos no-cero menores a 0.2 observados en la sección anterior.

Verificación: medir qué fracción de los saltos pequeños (< 0.2) cae exactamente en timestamps `hh:00:00`. Si la hipótesis es correcta, esa fracción debe ser muy alta.

In [ ]:
# Saltos pequeños: por debajo del deadband nominal
saltos_P_pequenos_mask = (saltos_P > 0) & (saltos_P < 0.2)
saltos_P_grandes_mask = saltos_P >= 0.2

n_pequenos = saltos_P_pequenos_mask.sum()
n_grandes = saltos_P_grandes_mask.sum()

# ¿Cuál es el TS asociado a cada salto?
# diff() le asigna el resultado a la fila n (no a la n-1), así que el TS del salto es df["TS"].iloc[n]
ts_saltos_pequenos = df.loc[saltos_P_pequenos_mask, "TS"]
ts_saltos_grandes = df.loc[saltos_P_grandes_mask, "TS"]

# ¿Cuántos caen exactamente en hh:00:00?
def es_hora_redonda(ts_series):
    return (ts_series.dt.minute == 0) & (ts_series.dt.second == 0)

pequenos_en_hora = es_hora_redonda(ts_saltos_pequenos).sum()
grandes_en_hora = es_hora_redonda(ts_saltos_grandes).sum()

print(f"Saltos pequeños (0 < |ΔP| < 0.2): {n_pequenos}")
print(f"  Caen en hh:00:00: {pequenos_en_hora} ({100*pequenos_en_hora/n_pequenos:.1f}%)")
print()
print(f"Saltos grandes (|ΔP| >= 0.2):    {n_grandes}")
print(f"  Caen en hh:00:00: {grandes_en_hora} ({100*grandes_en_hora/n_grandes:.1f}%)")
print()
print(f"Como referencia, en una distribución uniforme se esperaría que")
print(f"~{100/60:.1f}% de los timestamps cayeran en hh:00:00 (1 de cada 60 minutos).")

In [ ]:
# Chequeo de sanidad: ¿qué timestamps tienen los saltos pequeños?
# Mostramos algunos ejemplos concretos para verificar que diff() asigna correctamente.

# Encontrar los índices de los saltos pequeños
idx_saltos_pequenos = df.index[saltos_P_pequenos_mask]

print(f"Total de saltos pequeños: {len(idx_saltos_pequenos)}")
print(f"\nPrimeros 10 ejemplos (mostrando fila previa y fila del salto):\n")

for idx in idx_saltos_pequenos[:10]:
    ts_pre = df.loc[idx - 1, "TS"]
    p_pre = df.loc[idx - 1, "P"]
    ts_post = df.loc[idx, "TS"]
    p_post = df.loc[idx, "P"]
    delta = p_post - p_pre
    print(f"  {ts_pre} (P={p_pre})  →  {ts_post} (P={p_post})  |  ΔP={delta:+.5f}")

# Distribución de los segundos en los TS de saltos pequeños
print(f"\nDistribución de minutos en TS de saltos pequeños (top 10):")
minutos_pequenos = df.loc[saltos_P_pequenos_mask, "TS"].dt.minute.value_counts().head(10)
print(minutos_pequenos)

### 6.2 — Conclusiones de la cuantización del SCADA

**Hallazgos confirmados:**

- **P tiene cuantización por deadband nominal de 0.2** (operativamente conocido y empíricamente confirmado: moda de los saltos = 0.20).
- **El SCADA aplica heartbeat horario en P**: cada hora exacta fuerza una actualización del valor sin importar el deadband. La diferencia causada por ese heartbeat se materializa como salto pequeño (< 0.2) en el TS posterior al cambio de hora (`hh:01:00`). Verificación empírica: 704 de 710 saltos < 0.2 (99.2%) caen exactamente en `:01:00`, frente a 1.67% esperado bajo distribución uniforme.
- **Q no presenta cuantización SCADA detectable**: su distribución de saltos es continua sin pico dominante ni evidencia de heartbeat. La estabilidad inter-muestral de Q proviene de la física del flujo (caudal real estable durante minutos), no de filtrado del sensor.

**Implicaciones para el detector (capa 1 — suavizado):**

- **Ventana de suavizado ≥ 1 hora** es necesaria para absorber el heartbeat horario de P. Una ventana menor dejaría el heartbeat visible como pulso periódico en la señal suavizada, generando falsas alarmas de divergencia.
- **Ventana propuesta: 2 horas (120 muestras)** — cubre al menos un heartbeat completo con margen, y promedia suficientes saltos de deadband (~7 saltos típicos por ventana, dado que P cambia en 3.7% de las filas).
- El parámetro exacto se ajustará empíricamente en el notebook 02 sobre la base de este límite inferior.